# Budgerigar：复读内容保持评估

时间轴行为已经通过。本 notebook 检查模型输出是否对应当前输入句子，而不是目标声线的平均声学模板。它执行正确/打乱目标检索和输入消融对照。

In [ ]:
#@title 1. 更新项目与安装依赖
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [key for key in list(sys.modules) if key=='budgerigar' or key.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. 挂载 Drive 并定位 checkpoint
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
TARGET_SPEAKER='arctic_slt' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
RUN_DIR=WORK_ROOT/'checkpoints'/f'neural_echo_{TARGET_SPEAKER}_{FEATURE_FINGERPRINT}'
CHECKPOINT=RUN_DIR/'best.pt'
assert FEATURE_MANIFEST.is_file(),FEATURE_MANIFEST
assert CHECKPOINT.is_file(),CHECKPOINT
print(CHECKPOINT)

In [ ]:
#@title 3. 正确目标、打乱目标与输入消融评估
MAX_PAIRS=64 #@param {type:'integer'}
CANDIDATES=16 #@param {type:'integer'}
import json
from budgerigar.evaluate_content import evaluate_content
EVAL_DIR=RUN_DIR/'content_evaluation'
report=evaluate_content(CHECKPOINT,FEATURE_MANIFEST,EVAL_DIR,max_pairs=MAX_PAIRS,candidates=CANDIDATES)
print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 解释结果
print('content_pass =',report['content_pass'])
print('正确目标优势:',report['correct_vs_shuffled_margin'])
print('正确目标胜率:',report['correct_vs_shuffled_win_rate'])
print('Top-1 / chance:',report['retrieval_top1'],report['retrieval_chance'])
print('输入消融劣化:',report['input_ablation_degradation'])
if report['correct_vs_shuffled_margin']<=0.02: print('失败模式：输出不比随机目标更接近正确句子')
if report['retrieval_top1']<=report['retrieval_chance']*3: print('失败模式：句子检索接近随机')
if report['input_ablation_degradation']<=0.02: print('失败模式：移除输入后几乎不变，模型可能忽略内容')
print('报告：',EVAL_DIR/'content_report.json')

In [ ]:
#@title 5. 保存评估运行元数据
from budgerigar.experiment import write_run_metadata
metadata=write_run_metadata(EVAL_DIR/'run_metadata.json',FEATURE_MANIFEST,{'evaluation':'neural_echo_content','checkpoint':str(CHECKPOINT),'content_pass':report['content_pass'],'retrieval_top1':report['retrieval_top1']},repository=REPO_DIR)
print(metadata.read_text(encoding='utf-8'))